In [1]:
import requests, json
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
headers = {
    'User-Agent': 'Mozilla',
    'X-Requested-With': 'XMLHttpRequest',
}

In [3]:
ticker_url = 'https://www.macrotrends.net/assets/php/ticker_search_list.php'
r = requests.get(ticker_url, headers=headers)
ticker = json.loads(r.text)

In [4]:
tkr = {}
for t in range(len(ticker)):
    tmp = {}
    symbol = ticker[t]['s'].split('/')[0].replace('.','')
    issue_name = ticker[t]['n'].split(' - ')[1]
    if " ETF" not in issue_name:
        tmp['issue_name'] = issue_name
        tmp['site_ticker'] = ticker[t]['s']
        tkr[symbol] = tmp
tkr = pd.DataFrame.from_dict(tkr, orient='index')
tkr.head()

,issue_name,site_ticker
AAPL,Apple,AAPL/apple
AMZN,Amazon,AMZN/amazon
MSFT,Microsoft,MSFT/microsoft
GOOG,Alphabet,GOOG/alphabet
GOOGL,Alphabet,GOOGL/alphabet


In [5]:
tkr.loc['M']['site_ticker']

'M/macys'

In [75]:
def crawl_macrotrends(site_ticker, doc, freq='A'):
    stmt = {}
    url = 'https://www.macrotrends.net/stocks/charts/{}/{}?freq={}'.format(site_ticker, doc, freq)
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text)
    src = soup.find_all('script')[23]
    data = src.text.split('originalData = [{')[1].split('}];')[0]
    rst = data.split('},{')
    for r in rst:
        raw = json.loads('{'+r+'}')
        data_only = {}
        for k, v in raw.items():
            if k == 'field_name':
                title = v.split('>')[1].split('<')[0]
            try:
                data_only[pd.to_datetime(k)] = float(v)
            except:
                pass
        stmt[title] = data_only
    df = pd.DataFrame.from_dict(stmt, orient='index').iloc[:, ::-1]
    return df

In [77]:
crawl_macrotrends('AAPL/apple', 'income-statement', 'Q').tail(3)

,2005-03-31,2005-06-30,2005-09-30,2005-12-31,2006-03-31,2006-06-30,2006-09-30,2006-12-31,2007-03-31,2007-06-30,2007-09-30,2007-12-31,2008-03-31,2008-06-30,2008-09-30,2008-12-31,2009-03-31,2009-06-30,2009-09-30,2009-12-31,2010-03-31,2010-06-30,2010-09-30,2010-12-31,2011-03-31,2011-06-30,2011-09-30,2011-12-31,2012-03-31,2012-06-30,2012-09-30,2012-12-31,2013-03-31,2013-06-30,2013-09-30,2013-12-31,2014-03-31,2014-06-30,2014-09-30,2014-12-31,2015-03-31,2015-06-30,2015-09-30,2015-12-31,2016-03-31,2016-06-30,2016-09-30,2016-12-31,2017-03-31,2017-06-30,2017-09-30,2017-12-31,2018-03-31,2018-06-30,2018-09-30,2018-12-31,2019-03-31,2019-06-30,2019-09-30,2019-12-31,2020-03-31,2020-06-30
Shares Outstanding,23996.3100,24102.4800,23992.5800,24477.7900,24599.0400,24538.3000,24570.7300,24732.3200,24826.2900,24938.7900,24900.1800,25201.5100,25181.2100,25288.6800,25259.8900,25241.8300,25283.8000,25456.4800,25396.1400,25753.9200,25840.5800,25966.1100,25891.9400,26128.3100,26206.4300,26258.6800,26226.0600,26364.0200,26457.0000,26517.6500,26469.9300,26522.0800,26488.9800,25879.4200,26086.5400,25240.6400,24626.8000,24206.8400,24490.6500,23527.210,23339.4300,23092.4000,23172.280,22376.510,22163.5400,21891.1200,22001.1200,21311.980,21046.7500,20934.0000,21006.770,20631.1500,20273.9700,19706.440,20000.4400,19093.010,18802.5800,18405.520,18595.6500,17818.4200,17618.7600,17419.1500
Basic EPS,0.0129,0.0139,0.0186,0.0243,0.0175,0.0196,0.0229,0.0418,0.0318,0.0336,0.0371,0.0646,0.0425,0.0432,0.0975,0.0907,0.0650,0.0732,0.1004,0.1336,0.1211,0.1275,0.1682,0.2332,0.2318,0.2818,0.2550,0.5011,0.4446,0.3364,0.3129,0.4975,0.3629,0.2675,0.3021,0.5200,0.4175,0.3225,0.3625,0.770,0.5850,0.4650,0.500,0.825,0.4775,0.3575,0.4275,0.845,0.5275,0.4200,0.525,0.9800,0.6875,0.590,0.7450,1.055,0.6175,0.550,0.7700,1.2600,0.6450,0.6525
EPS - Earnings Per Share,0.0121,0.0132,0.0175,0.0232,0.0168,0.0193,0.0218,0.0407,0.0311,0.0329,0.0357,0.0629,0.0414,0.0425,0.0954,0.0893,0.0639,0.0718,0.0993,0.1311,0.1189,0.1254,0.1657,0.2296,0.2286,0.2782,0.2521,0.4954,0.4393,0.3329,0.3100,0.4932,0.3604,0.2675,0.2989,0.5175,0.4150,0.3200,0.3600,0.765,0.5825,0.4625,0.495,0.820,0.4750,0.3550,0.4275,0.840,0.5250,0.4175,0.520,0.9725,0.6825,0.585,0.7375,1.045,0.6150,0.545,0.7675,1.2475,0.6375,0.6450


In [81]:
def financials_raw(site_ticker, freq='A'):
    df = pd.DataFrame()
    for d in ['balance-sheet','cash-flow-statement','income-statement','financial-ratios']:
        d = crawl_macrotrends(site_ticker, d, freq=freq)
        df = pd.concat([df, d], axis=0)
    return df

In [82]:
fn = financials_raw('AAPL/apple', 'Q')

In [83]:
fn

,2005-03-31,2005-06-30,2005-09-30,2005-12-31,2006-03-31,2006-06-30,2006-09-30,2006-12-31,2007-03-31,2007-06-30,2007-09-30,2007-12-31,2008-03-31,2008-06-30,2008-09-30,2008-12-31,2009-03-31,2009-06-30,2009-09-30,2009-12-31,2010-03-31,2010-06-30,2010-09-30,2010-12-31,2011-03-31,2011-06-30,2011-09-30,2011-12-31,2012-03-31,2012-06-30,2012-09-30,2012-12-31,2013-03-31,2013-06-30,2013-09-30,2013-12-31,2014-03-31,2014-06-30,2014-09-30,2014-12-31,2015-03-31,2015-06-30,2015-09-30,2015-12-31,2016-03-31,2016-06-30,2016-09-30,2016-12-31,2017-03-31,2017-06-30,2017-09-30,2017-12-31,2018-03-31,2018-06-30,2018-09-30,2018-12-31,2019-03-31,2019-06-30,2019-09-30,2019-12-31,2020-03-31,2020-06-30
Cash On Hand,7057.0000,7526.0000,8261.0000,8707.0000,8226.0000,9176.0000,10110.0000,11869.0000,12577.0000,13767.0000,15386.0000,18448.0000,19448.0000,20774.0000,22111.0000,25647.0000,25013.0000,24222.0000,23464.0000,24796.0000,23155.0000,24288.0000,25620.0000,26977.0000,29234.0000,28395.0000,25952.0000,30156.0000,28538.0000,27654.0000,29129.0000,39820.0000,39137.0000,42606.0000,40546.0000,40711.0000,41350.0000,37805.0000,25077.0000,32463.0000,33096.0000,34703.0000,41601.0000,38074.0000,55283.0000,61756.0000,67155.0000,60452.0000,67101.0000,76759.0000,74181.0000,77153.0000,87940.0000,70970.0000,66301.0000,86427.0000,80092.0000,94614.0000,100557.0000,107162.0000,94051.0000,93025.0000
Receivables,888.0000,827.0000,895.0000,1331.0000,861.0000,1089.0000,1252.0000,1621.0000,928.0000,1410.0000,1637.0000,1939.0000,1593.0000,1603.0000,2422.0000,2196.0000,1932.0000,2686.0000,5057.0000,3090.0000,2886.0000,6399.0000,9924.0000,10874.0000,11095.0000,11471.0000,11717.0000,16484.0000,13769.0000,14298.0000,18692.0000,21534.0000,13336.0000,13453.0000,20641.0000,25198.0000,15820.0000,16841.0000,27219.0000,29976.0000,18164.0000,19907.0000,30343.0000,24621.0000,19824.0000,19042.0000,29299.0000,27977.0000,20612.0000,22632.0000,35673.0000,50899.0000,22408.0000,26367.0000,48995.0000,36981.0000,26278.0000,26474.0000,45804.0000,39946.0000,30677.0000,32075.0000
Inventory,164.0000,193.0000,165.0000,244.0000,204.0000,213.0000,270.0000,303.0000,208.0000,251.0000,346.0000,459.0000,364.0000,545.0000,509.0000,396.0000,312.0000,380.0000,455.0000,576.0000,638.0000,942.0000,1051.0000,885.0000,930.0000,889.0000,776.0000,1236.0000,1102.0000,1122.0000,791.0000,1455.0000,1245.0000,1697.0000,1764.0000,2122.0000,1829.0000,1594.0000,2111.0000,2283.0000,2396.0000,2042.0000,2349.0000,2451.0000,2281.0000,1831.0000,2132.0000,2712.0000,2910.0000,3146.0000,4855.0000,4421.0000,7662.0000,5936.0000,3956.0000,4988.0000,4884.0000,3355.0000,4106.0000,4097.0000,3334.0000,3978.0000
Other Current Assets,601.0000,496.0000,648.0000,1409.0000,1536.0000,1522.0000,2270.0000,2223.0000,1676.0000,2630.0000,3805.0000,4350.0000,4271.0000,3945.0000,3920.0000,5311.0000,5057.0000,6151.0000,1444.0000,3690.0000,4515.0000,3188.0000,3447.0000,3467.0000,4055.0000,4251.0000,4529.0000,4958.0000,5050.0000,6560.0000,6458.0000,6644.0000,6377.0000,7270.0000,6882.0000,8574.0000,7528.0000,7825.0000,9806.0000,13635.0000,9094.0000,9291.0000,15085.0000,11073.0000,10204.0000,11132.0000,8283.0000,12191.0000,11367.0000,10338.0000,13936.0000,11337.0000,12043.0000,12488.0000,12087.0000,12432.0000,12092.0000,10530.0000,12352.0000,12026.0000,15691.0000,10987.0000
Total Current Assets,9007.0000,9376.0000,10300.0000,12162.0000,11286.0000,12491.0000,14509.0000,16664.0000,16029.0000,18745.0000,21956.0000,26189.0000,26736.0000,27998.0000,30006.0000,35163.0000,33853.0000,35170.0000,31555.0000,33332.0000,32336.0000,36033.0000,41678.0000,43927.0000,46997.0000,46898.0000,44988.0000,54771.0000,50712.0000,51943.0000,57653.0000,72348.0000,63337.0000,68219.0000,73286.0000,80347.0000,70541.0000,67949.0000,68531.0000,83403.0000,67891.0000,70953.0000,89378.0000,76219.0000,87592.0000,93761.0000,106869.0000,103332.0000,101990.0000,112875.0000,128645.0000,143810.0000,130053.0000,115761.0000,131339.0000,140828.0000,123346.0000,134973.0000,162819.0000,163231.0000,143753

In [92]:
def financials_ttm(site_ticker):
    df = pd.DataFrame()
    for d in ['income-statement','cash-flow-statement']:
        d = crawl_macrotrends(site_ticker, d, freq='Q')
        df = pd.concat([df, d], axis=0)
    df = df.rolling(4, axis=1).sum()
    for d in ['balance-sheet']:
        d = crawl_macrotrends(site_ticker, d, freq='Q')
        df = pd.concat([df, d], axis=0)
    return df

In [93]:
fn_ttm = financials_ttm('AAPL/apple')

In [94]:
fn_ttm.iloc[:20,-4]

Revenue                               260174.0000
Cost Of Goods Sold                    161782.0000
Gross Profit                           98392.0000
Research And Development Expenses      16217.0000
SG&A Expenses                          18245.0000
Operating Expenses                    196244.0000
Operating Income                       63930.0000
Total Non-Operating Income/Expense      1807.0000
Pre-Tax Income                         65737.0000
Income Taxes                           10481.0000
Income After Taxes                     55256.0000
Income From Continuous Operations      55256.0000
Net Income                             55256.0000
EBITDA                                 76477.0000
EBIT                                   63930.0000
Basic Shares Outstanding               74393.4300
Shares Outstanding                     74896.7600
Basic EPS                                  2.9925
EPS - Earnings Per Share                   2.9725
Net Income/Loss                        55256.0000


In [19]:
converter = {
    # title: [영문명, multiplier, order, ttm산출방법]
    'Revenue': ['Revenue', 1000000, 'DESC', 'SUM'],
    'Cost Of Goods Sold': ['COGS', 1000000, 'DESC', 'SUM'],
    'Gross Profit': ['Gross Profit', 1000000, 'DESC', 'SUM'],
    'Operating Income': ['Operating Income', 1000000, 'DESC', 'SUM'],
    'Net Income': ['Net Income', 1000000, 'DESC', 'SUM'],
    'EBITDA': ['EBITDA', 1000000, 'DESC', 'SUM'],
    'Shares Outstanding': ['Shares', 1000000, 'DESC', 'LAST'],
    'EPS - Earnings Per Share': ['EPS', 1, 'DESC', 'SUM'],

    'Total Assets': ['Asset', 1000000, 'DESC', 'LAST'],
    'Total Current Assets': ['Current Asset', 1000000, 'DESC', 'LAST'],
    'Total Liabilities': ['Liabaility', 1000000, 'ASC', 'LAST'],
    'Total Current Liabilities': ['Current Liabaility', 1000000, 'ASC', 'LAST'],
    'Long Term Debt': ['LT Debt', 1000000, 'DESC', 'LAST'],
    'Share Holder Equity': ['Equity', 1000000, 'DESC', 'LAST'],

    'Cash Flow From Operating Activities': ['CFO', 1000000, 'DESC', 'SUM'],
    'Cash Flow From Investing Activities': ['CFI', 1000000, 'DESC', 'SUM'],
    'Cash Flow From Financial Activities': ['CFF', 1000000, 'DESC', 'SUM'],
    'Net Cash Flow': ['Net Cash Flow', 1000000, 'DESC', 'SUM'],
    'Common Stock Dividends Paid': ['Dividend', 1000000, 'DESC', 'SUM'],
    
    'Current Ratio': ['Current Ratio', 1, 'DESC', 'LAST'],
    'Debt/Equity Ratio': ['Debt/Equity', 1, 'DESC', 'LAST'],
    'ROE - Return On Equity': ['ROE', 1, 'DESC', 'SUM'],
    'ROA - Return On Assets': ['ROA', 1, 'DESC', 'SUM'],
    'ROI - Return On Investment': ['ROI', 1, 'DESC', 'SUM'],
    'Book Value Per Share': ['BPS', 1, 'DESC', 'LAST'],
    'Operating Cash Flow Per Share': ['CF/S', 1, 'DESC', 'LAST'],
    'Free Cash Flow Per Share': ['FCF/S', 1, 'DESC', 'LAST'],
}

In [20]:
fn.iloc[:,:4]

,2020-06-30,2020-03-31,2019-12-31,2019-09-30
Cash On Hand,93025.0000,94051.0000,107162.0000,100557.0000
Receivables,32075.0000,30677.0000,39946.0000,45804.0000
Inventory,3978.0000,3334.0000,4097.0000,4106.0000
Other Current Assets,10987.0000,15691.0000,12026.0000,12352.0000
Total Current Assets,140065.0000,143753.0000,163231.0000,162819.0000
...,...,...,...,...
ROI - Return On Investment,6.7655,6.7154,12.1768,7.5076
Book Value Per Share,4.2182,4.5343,5.1044,5.0913
Operating Cash Flow Per Share,0.9626,0.7749,1.7126,1.0432
Free Cash Flow Per Share,0.8701,0.6684,1.5944,0.8982


In [21]:
renamer = {}
for k,v in converter.items():
  renamer[k] = v[0]
renamer

{'Book Value Per Share': 'BPS',
 'Cash Flow From Financial Activities': 'CFF',
 'Cash Flow From Investing Activities': 'CFI',
 'Cash Flow From Operating Activities': 'CFO',
 'Common Stock Dividends Paid': 'Dividend',
 'Cost Of Goods Sold': 'COGS',
 'Current Ratio': 'Current Ratio',
 'Debt/Equity Ratio': 'Debt/Equity',
 'EBITDA': 'EBITDA',
 'EPS - Earnings Per Share': 'EPS',
 'Free Cash Flow Per Share': 'FCF/S',
 'Gross Profit': 'Gross Profit',
 'Long Term Debt': 'LT Debt',
 'Net Cash Flow': 'Net Cash Flow',
 'Net Income': 'Net Income',
 'Operating Cash Flow Per Share': 'CF/S',
 'Operating Income': 'Operating Income',
 'ROA - Return On Assets': 'ROA',
 'ROE - Return On Equity': 'ROE',
 'ROI - Return On Investment': 'ROI',
 'Revenue': 'Revenue',
 'Share Holder Equity': 'Equity',
 'Shares Outstanding': 'Shares',
 'Total Assets': 'Asset',
 'Total Current Assets': 'Current Asset',
 'Total Current Liabilities': 'Current Liabaility',
 'Total Liabilities': 'Liabaility'}

In [26]:
rounder = {}
for k,v in converter.items():
  rounder[k] = 0 if v[1] == 1000000 else 2
rounder

{'Book Value Per Share': 2,
 'Cash Flow From Financial Activities': 0,
 'Cash Flow From Investing Activities': 0,
 'Cash Flow From Operating Activities': 0,
 'Common Stock Dividends Paid': 0,
 'Cost Of Goods Sold': 0,
 'Current Ratio': 2,
 'Debt/Equity Ratio': 2,
 'EBITDA': 0,
 'EPS - Earnings Per Share': 2,
 'Free Cash Flow Per Share': 2,
 'Gross Profit': 0,
 'Long Term Debt': 0,
 'Net Cash Flow': 0,
 'Net Income': 0,
 'Operating Cash Flow Per Share': 2,
 'Operating Income': 0,
 'ROA - Return On Assets': 2,
 'ROE - Return On Equity': 2,
 'ROI - Return On Investment': 2,
 'Revenue': 0,
 'Share Holder Equity': 0,
 'Shares Outstanding': 0,
 'Total Assets': 0,
 'Total Current Assets': 0,
 'Total Current Liabilities': 0,
 'Total Liabilities': 0}

In [27]:
df = fn.loc[converter.keys()].iloc[:,[3,2,1,0]]
df = df.round(rounder)
df.rename(index=renamer, inplace=True)
df

,2019-09-30,2019-12-31,2020-03-31,2020-06-30
Revenue,64040.0000,91819.0000,58313.0000,59685.0000
COGS,39727.0000,56602.0000,35943.0000,37005.0000
Gross Profit,24313.0000,35217.0000,22370.0000,22680.0000
Operating Income,15625.0000,25569.0000,12853.0000,13091.0000
Net Income,13686.0000,22236.0000,11249.0000,11253.0000
EBITDA,18804.0000,28385.0000,15639.0000,15843.0000
Shares,18595.6500,17818.4200,17618.7600,17419.1500
EPS,0.7675,1.2475,0.6375,0.6450
Asset,338516.0000,340618.0000,320400.0000,317344.0000
Current Asset,162819.0000,163231.0000,143753.0000,140065.0000


In [24]:
for k,v in converter.items():
  print(v[0], fn.loc[k, '2020/06'])

Revenue 2020-06-30    59685.0
Name: Revenue, dtype: float64
COGS 2020-06-30    37005.0
Name: Cost Of Goods Sold, dtype: float64
Gross Profit 2020-06-30    22680.0
Name: Gross Profit, dtype: float64
Operating Income 2020-06-30    13091.0
Name: Operating Income, dtype: float64
Net Income 2020-06-30    11253.0
Name: Net Income, dtype: float64
EBITDA 2020-06-30    15843.0
Name: EBITDA, dtype: float64
Shares 2020-06-30    17419.15
Name: Shares Outstanding, dtype: float64
EPS 2020-06-30    0.645
Name: EPS - Earnings Per Share, dtype: float64
Asset 2020-06-30    317344.0
Name: Total Assets, dtype: float64
Current Asset 2020-06-30    140065.0
Name: Total Current Assets, dtype: float64
Liabaility 2020-06-30    245062.0
Name: Total Liabilities, dtype: float64
Current Liabaility 2020-06-30    95318.0
Name: Total Current Liabilities, dtype: float64
LT Debt 2020-06-30    94048.0
Name: Long Term Debt, dtype: float64
Equity 2020-06-30    72282.0
Name: Share Holder Equity, dtype: float64
CFO 2020-06-3

In [39]:
ttm = fn.iloc[:, ::-1]

In [40]:
ttm.rolling(4, axis=1).sum()

,2005-03-31,2005-06-30,2005-09-30,2005-12-31,2006-03-31,2006-06-30,2006-09-30,2006-12-31,2007-03-31,2007-06-30,2007-09-30,2007-12-31,2008-03-31,2008-06-30,2008-09-30,2008-12-31,2009-03-31,2009-06-30,2009-09-30,2009-12-31,2010-03-31,2010-06-30,2010-09-30,2010-12-31,2011-03-31,2011-06-30,2011-09-30,2011-12-31,2012-03-31,2012-06-30,2012-09-30,2012-12-31,2013-03-31,2013-06-30,2013-09-30,2013-12-31,2014-03-31,2014-06-30,2014-09-30,2014-12-31,2015-03-31,2015-06-30,2015-09-30,2015-12-31,2016-03-31,2016-06-30,2016-09-30,2016-12-31,2017-03-31,2017-06-30,2017-09-30,2017-12-31,2018-03-31,2018-06-30,2018-09-30,2018-12-31,2019-03-31,2019-06-30,2019-09-30,2019-12-31,2020-03-31,2020-06-30
Cash On Hand,NaN,NaN,NaN,31551.0000,32720.0000,34370.0000,36219.0000,39381.0000,43732.0000,48323.0000,53599.0000,60178.0000,67049.0000,74056.0000,80781.0000,87980.0000,93545.0000,96993.0000,98346.0000,97495.0000,95637.0000,95703.0000,97859.0000,100040.0000,106119.0000,110226.0000,110558.0000,113737.0000,113041.0000,112300.0000,115477.0000,125141.0000,135740.0000,150692.0000,162109.0000,163000.0000,165213.0000,160412.0000,144943.0000,136695.0000,128441.0000,125339.0000,141863.0000,147474.0000,169661.0000,196714.0000,222268.0000,244646.0000,256464.0000,271467.0000,278493.0000,295194.0000,316033.0000,310244.0000,302364.0000,311638.0000,303790.0000,327434.0000,361690.0000,382425.0000,396384.0000,394795.0000
Receivables,NaN,NaN,NaN,3941.0000,3914.0000,4176.0000,4533.0000,4823.0000,4890.0000,5211.0000,5596.0000,5914.0000,6579.0000,6772.0000,7557.0000,7814.0000,8153.0000,9236.0000,11871.0000,12765.0000,13719.0000,17432.0000,22299.0000,30083.0000,38292.0000,43364.0000,45157.0000,50767.0000,53441.0000,56268.0000,63243.0000,68293.0000,67860.0000,67015.0000,68964.0000,72628.0000,75112.0000,78500.0000,85078.0000,89856.0000,92200.0000,95266.0000,98390.0000,93035.0000,94695.0000,93830.0000,92786.0000,96142.0000,96930.0000,100520.0000,106894.0000,129816.0000,131612.0000,135347.0000,148669.0000,134751.0000,138621.0000,138728.0000,135537.0000,138502.0000,142901.0000,148502.0000
Inventory,NaN,NaN,NaN,766.0000,806.0000,826.0000,931.0000,990.0000,994.0000,1032.0000,1108.0000,1264.0000,1420.0000,1714.0000,1877.0000,1814.0000,1762.0000,1597.0000,1543.0000,1723.0000,2049.0000,2611.0000,3207.0000,3516.0000,3808.0000,3755.0000,3480.0000,3831.0000,4003.0000,4236.0000,4251.0000,4470.0000,4613.0000,5188.0000,6161.0000,6828.0000,7412.0000,7309.0000,7656.0000,7817.0000,8384.0000,8832.0000,9070.0000,9238.0000,9123.0000,8912.0000,8695.0000,8956.0000,9585.0000,10900.0000,13623.0000,15332.0000,20084.0000,22874.0000,21975.0000,22542.0000,19764.0000,17183.0000,17333.0000,16442.0000,14892.0000,15515.0000
Other Current Assets,NaN,NaN,NaN,3154.0000,4089.0000,5115.0000,6737.0000,7551.0000,7691.0000,8799.0000,10334.0000,12461.0000,15056.0000,16371.0000,16486.0000,17447.0000,18233.0000,20439.0000,17963.0000,16342.0000,15800.0000,12837.0000,14840.0000,14617.0000,14157.0000,15220.0000,16302.0000,17793.0000,18788.0000,21097.0000,23026.0000,24712.0000,26039.0000,26749.0000,27173.0000,29103.0000,30254.0000,30809.0000,33733.0000,38794.0000,40360.0000,41826.0000,47105.0000,44543.0000,45653.0000,47494.0000,40692.0000,41810.0000,42973.0000,42179.0000,47832.0000,46978.0000,47654.0000,49804.0000,47955.0000,49050.0000,49099.0000,47141.0000,47406.0000,47000.0000,50599.0000,51056.0000
Total Current Assets,NaN,NaN,NaN,40845.0000,43124.0000,46239.0000,50448.0000,54950.0000,59693.0000,65947.0000,73394.0000,82919.0000,93626.0000,102879.0000,110929.0000,119903.0000,127020.0000,134192.0000,135741.0000,133910.0000,132393.0000,133256.0000,143379.0000,153974.0000,168635.0000,179500.0000,182810.0000,193654.0000,197369.0000,202414.0000,215079.0000,232656.0000,245281.0000,261557.0000,277190.0000,285189.0000,292393.0000,292123.0000,287368.0000,290424.0000,287774.0000,290778.0000,311625.0000,304441.0000,324142.0000,346950.0000,364441.0000,391554.0000,405952.0000,425066.0000,446842.0000,487320.0000,515383.0000,518269.0000,5209